In [3]:
"""
train.py — PPO training script for the ColoredChestKuka task (Task 1).

Usage
-----
Train with the advanced reward shaping (recommended):

    python train.py --reward advanced --steps 500000 --seed 42

Train with the basic reward for comparison:

    python train.py --reward basic --steps 500000 --seed 42

Train both configurations in sequence:

    python train.py --both --steps 500000

All artefacts (model weights, metric logs, training curves) are written to the
``results/`` directory.

Theory
------
We frame the KUKA reaching task as a continuous-action MDP and solve it with
Proximal Policy Optimisation (PPO; Schulman et al., 2017).

    * State (10-D): end-effector position (3), target chest top-centre (3),
      one-hot target color (3), episode progress (1).
    * Action (3-D continuous): Cartesian delta [Δx, Δy, Δz] clipped to
      ±action_scale = ±0.05 m.
    * Reward: see ColoredChestKukaEnv docstring for "basic" vs "advanced".

PPO was chosen over:
    - Q-learning / DQN: incompatible with a continuous action space.
    - DDPG / SAC: also valid, but PPO's clipped surrogate objective provides
      more stable on-policy updates for this relatively low-dimensional task
      and is the approach suggested in the project brief.
    - Imitation learning: no expert demonstrations available.

Key hyperparameters (all others remain at SB3 defaults):
    - n_steps = 2048   : rollout length per environment before each update
    - batch_size = 256  : mini-batch size for the PPO surrogate loss
    - n_epochs = 10     : SGD epochs per PPO update
    - learning_rate = 3e-4
    - gamma = 0.99      : discount factor
    - ent_coef = 0.001  : entropy bonus to encourage exploration
    - clip_range = 0.2  : PPO clipping parameter ε
    - policy = "MlpPolicy" with [256, 256] hidden layers
"""

import argparse
import json
import os
import sys
import time

import numpy as np

# Make src/ importable when running from the project root
sys.path.insert(0, os.path.dirname(__file__))

from colored_chest_kuka_env import ColoredChestKukaEnv  # noqa: E402

try:
    import gymnasium as gym
    from stable_baselines3 import PPO
    from stable_baselines3.common.env_util import make_vec_env
    from stable_baselines3.common.monitor import Monitor
    from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
    from callbacks import EpisodeMetricsCallback, CheckpointWithMetricsCallback
    _HAS_SB3 = True
except ImportError as e:
    _HAS_SB3 = False
    _IMPORT_ERROR = str(e)


# ---------------------------------------------------------------------------
# Default hyperparameters
# ---------------------------------------------------------------------------

PPO_HYPERPARAMS = dict(
    policy="MlpPolicy",
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    learning_rate=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.001,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=dict(net_arch=[256, 256]),
    verbose=1,
)

RESULTS_DIR = os.path.join(os.path.dirname(__file__), "..", "results")
MODELS_DIR = os.path.join(os.path.dirname(__file__), "..", "models")


# ---------------------------------------------------------------------------
# Training function
# ---------------------------------------------------------------------------

def train(
    reward_type: str = "advanced",
    total_timesteps: int = 500_000,
    n_envs: int = 4,
    seed: int = 42,
    checkpoint_freq: int = 50_000,
) -> None:
    """
    Train a PPO agent on the ColoredChestKuka environment.

    Parameters
    ----------
    reward_type : str
        ``"basic"`` or ``"advanced"``.  Passed to :class:`ColoredChestKukaEnv`.
    total_timesteps : int
        Total number of environment steps to train for.
    n_envs : int
        Number of parallel environments (SubprocVecEnv).
    seed : int
        Random seed for reproducibility.
    checkpoint_freq : int
        Save a model checkpoint every this many steps.
    """
    if not _HAS_SB3:
        raise ImportError(
            f"stable-baselines3 is required to run training. "
            f"Install with:  pip install stable-baselines3\n"
            f"Import error was: {_IMPORT_ERROR}"
        )

    print(f"\n{'='*60}")
    print(f"  PPO Training — reward_type='{reward_type}'")
    print(f"  total_timesteps={total_timesteps:,}   n_envs={n_envs}   seed={seed}")
    print(f"{'='*60}\n")

    os.makedirs(RESULTS_DIR, exist_ok=True)
    os.makedirs(MODELS_DIR, exist_ok=True)

    # ------------------------------------------------------------------
    # Build vectorised environment
    # ------------------------------------------------------------------
    def _make_env(rank: int):
        """Factory that creates one monitored env with a unique seed."""
        def _init():
            env = ColoredChestKukaEnv(
                reward_type=reward_type,
                render_mode=None,   # headless for training
                max_steps=150,
            )
            env = Monitor(env)      # wraps env so SB3 sees episode stats
            env.reset(seed=seed + rank)
            return env
        return _init

    # Use DummyVecEnv (single-process) for safety on all platforms.
    # Switch to SubprocVecEnv for genuine multi-core speedup on Linux.
    vec_env = DummyVecEnv([_make_env(i) for i in range(n_envs)])

    # ------------------------------------------------------------------
    # Instantiate PPO model
    # ------------------------------------------------------------------
    model = PPO(
        env=vec_env,
        seed=seed,
        tensorboard_log=os.path.join(RESULTS_DIR, "tensorboard_logs"),
        **PPO_HYPERPARAMS,
    )

    # ------------------------------------------------------------------
    # Callbacks
    # ------------------------------------------------------------------
    metrics_cb = EpisodeMetricsCallback(verbose=1, window=20)

    checkpoint_cb = CheckpointWithMetricsCallback(
        save_freq=checkpoint_freq,
        save_path=os.path.join(RESULTS_DIR, "training_logs"),
        metrics_callback=metrics_cb,
        name_prefix=f"ppo_{reward_type}",
        verbose=1,
    )

    # ------------------------------------------------------------------
    # Train
    # ------------------------------------------------------------------
    t0 = time.time()
    model.learn(
        total_timesteps=total_timesteps,
        callback=[metrics_cb, checkpoint_cb],
        progress_bar=True,
    )
    elapsed = time.time() - t0
    print(f"\nTraining wall-clock time: {elapsed/60:.1f} min")

    # ------------------------------------------------------------------
    # Save final model and metrics
    # ------------------------------------------------------------------
    final_model_path = os.path.join(MODELS_DIR, f"ppo_{reward_type}_final")
    model.save(final_model_path)
    print(f"Final model saved to {final_model_path}.zip")

    metrics_path = os.path.join(RESULTS_DIR, f"ppo_{reward_type}_metrics.json")
    with open(metrics_path, "w") as fh:
        json.dump(metrics_cb.to_dict(), fh, indent=2)
    print(f"Metrics saved to {metrics_path}")

    vec_env.close()
    print("\nDone.")


# ---------------------------------------------------------------------------
# CLI entry point
# ---------------------------------------------------------------------------

def _build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(
        description="Train a PPO agent on the ColoredChestKuka environment.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument(
        "--reward",
        choices=["basic", "advanced"],
        default="advanced",
        help="Reward formulation to use.",
    )
    p.add_argument(
        "--steps",
        type=int,
        default=500_000,
        help="Total training timesteps.",
    )
    p.add_argument(
        "--n-envs",
        type=int,
        default=4,
        help="Number of parallel environments.",
    )
    p.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Random seed.",
    )
    p.add_argument(
        "--checkpoint-freq",
        type=int,
        default=50_000,
        help="Save a checkpoint every N steps.",
    )
    p.add_argument(
        "--both",
        action="store_true",
        help="Train both reward types sequentially (ignores --reward).",
    )
    return p


if __name__ == "__main__":
    args = _build_parser().parse_args()

    configs = (
        ["basic", "advanced"]
        if args.both
        else [args.reward]
    )

    for reward_type in configs:
        train(
            reward_type=reward_type,
            total_timesteps=args.steps,
            n_envs=args.n_envs,
            seed=args.seed,
            checkpoint_freq=args.checkpoint_freq,
        )

NameError: name '__file__' is not defined